# 01 ネイタルチャート スキーマ検証

エンジンの出力を確認し、FastAPI レスポンススキーマに整形する工程を検証する。

**テストデータ**: ジョン・レノン 1940-10-09 18:30 Liverpool, UK
（パブリックドメインの有名チャート: 検証サイトと照合しやすい）

In [ ]:
import sys
sys.path.insert(0, "..")

from datetime import datetime
from engine.calculator import WesternAstrologyEngine
from engine.dignity import calc_all_dignities, calc_ruler_analysis, get_dignity, DIGNITY_SCORE

engine = WesternAstrologyEngine()
print("Engine initialized")

## 1. ネイタルチャート計算

In [ ]:
# ジョン・レノン 1940-10-09 18:30 Liverpool (lat=53.4084, lon=-2.9916, tz=Europe/London)
birth_dt = datetime(1940, 10, 9, 18, 30)
lat, lon, tz_str = 53.4084, -2.9916, "Europe/London"

natal = engine.calc_natal(birth_dt, lat, lon, tz_str, house_system="P")

print(f"JD: {natal['jd']:.4f}")
print(f"天体数: {len(natal['planets'])} ({', '.join(natal['planets'].keys())})")
print(f"アスペクト数: {len(natal['aspects'])}")

## 2. 天体位置の確認

期待値（参照: Astro.com）:
- Sun: 16°Libra, Moon: 3°Aquarius
- Mercury: 8°Scorpio, Venus: 3°Virgo
- ASC: 18°Aries

In [ ]:
import pandas as pd

rows = []
for name, p in natal["planets"].items():
    rows.append({
        "天体": name,
        "度数": p["degree_str"],
        "黄経": f"{p['longitude']:.2f}°",
        "ハウス": p["house"],
        "逆行": "R" if p["retrograde"] else "",
    })

df = pd.DataFrame(rows)
df

## 3. ハウスカスプ

In [ ]:
from engine.calculator import ZODIAC_SIGNS

cusps = natal["houses"]["cusps"]
for i, c in enumerate(cusps, 1):
    sign = ZODIAC_SIGNS[int(c / 30)]
    deg  = c % 30
    print(f"H{i:2d}: {int(deg):2d}°{int((deg%1)*60):02d}' {sign} ({c:.2f}°)")
print(f"ASC: {natal['houses']['asc']:.2f}°")
print(f"MC : {natal['houses']['mc']:.2f}°")

## 4. ディグニティ（品位）計算

In [ ]:
dignities = calc_all_dignities(natal["planets"])

df_dig = pd.DataFrame(dignities)[["planet", "sign_jp", "house", "dignity", "score", "retrograde"]]
df_dig = df_dig.sort_values("score", ascending=False)
df_dig

## 5. ルーラー分析

In [ ]:
rulers = calc_ruler_analysis(natal)

df_rul = pd.DataFrame(rulers)
df_rul[["house_num", "cusp_sign", "ruler", "ruler_sign_jp", "ruler_house", "dignity", "retrograde"]]

## 6. サマリー（エレメント・モダリティ・ステリウム）

In [ ]:
import json

summary = engine.calc_summary(natal)

print("--- エレメント ---")
for k, v in summary["element_count_jp"].items():
    print(f"  {k}: {v}")

print("\n--- モダリティ ---")
for k, v in summary["modality_count_jp"].items():
    print(f"  {k}: {v}")

print(f"\n太陽: {summary['sun_sign_jp']} / 月: {summary['moon_sign_jp']} / ASC: {summary['asc_sign_jp']}")

print("\n--- ステリウム ---")
if summary["stelliums"]:
    for s in summary["stelliums"]:
        print(f"  {s['label']} ({s['type']}): {', '.join(s['planets'])} ({s['count']}天体)")
else:
    print("  なし")

## 7. アスペクト一覧

In [ ]:
df_asp = pd.DataFrame(natal["aspects"])[["planet1_jp", "planet2_jp", "aspect_jp", "orb", "type", "applying_jp"]]
df_asp = df_asp.sort_values("orb")
df_asp.head(20)

## 8. APIスキーマ整形

FastAPI `POST /chart` のレスポンスを想定して組み立てる。
- `planets`: dignity を各天体オブジェクトにマージ
- `summary`: エレメント・モダリティ・ステリウム
- `rulers`: 各ハウスの支配星分析
- `total_score`: dignity スコアの合計
- `score_label`: スコアから簡易ラベル

In [ ]:
def build_api_response(
    natal: dict,
    summary: dict,
    dignities: list,
    rulers: list,
    birth_dt: datetime,
    lat: float,
    lon: float,
) -> dict:
    """エンジン出力 → FastAPI レスポンス dict に整形"""

    # dignity を planet にマージ
    dig_map = {d["planet"]: d for d in dignities}
    planets_out = {}
    for name, p in natal["planets"].items():
        entry = dict(p)
        d = dig_map.get(name)
        if d:
            entry["dignity"]       = d["dignity"]
            entry["dignity_score"] = d["score"]
        else:
            entry["dignity"]       = None
            entry["dignity_score"] = None
        planets_out[name] = entry

    # 総合スコア（dignityスコア合計）
    total_score = sum(d["score"] for d in dignities)

    if total_score >= 10:
        score_label = "大吉"
    elif total_score >= 3:
        score_label = "吉"
    elif total_score >= -3:
        score_label = "中"
    elif total_score >= -10:
        score_label = "凶"
    else:
        score_label = "大凶"

    return {
        "birth_dt":    birth_dt.isoformat(),
        "lat":         lat,
        "lon":         lon,
        "tz":          natal["tz"],
        "jd":          natal["jd"],
        "planets":     planets_out,
        "houses":      {
            "cusps":   natal["houses"]["cusps"],
            "asc":     natal["houses"]["asc"],
            "mc":      natal["houses"]["mc"],
        },
        "aspects":     natal["aspects"],
        "summary":     {
            "sun_sign":         summary["sun_sign"],
            "moon_sign":        summary["moon_sign"],
            "asc_sign":         summary["asc_sign"],
            "sun_sign_jp":      summary["sun_sign_jp"],
            "moon_sign_jp":     summary["moon_sign_jp"],
            "asc_sign_jp":      summary["asc_sign_jp"],
            "element_count":    summary["element_count"],
            "element_count_jp": summary["element_count_jp"],
            "modality_count":   summary["modality_count"],
            "modality_count_jp":summary["modality_count_jp"],
            "stelliums":        summary["stelliums"],
        },
        "rulers":      rulers,
        "total_score": total_score,
        "score_label": score_label,
    }


response = build_api_response(natal, summary, dignities, rulers, birth_dt, lat, lon)
print("レスポンスキー:", list(response.keys()))
print(f"total_score: {response['total_score']}  score_label: {response['score_label']}")

## 9. スキーマ構造の確認

In [ ]:
# 太陽の完全オブジェクト確認
print("--- planets['Sun'] ---")
print(json.dumps(response["planets"]["Sun"], ensure_ascii=False, indent=2))

In [ ]:
# サマリー確認
print("--- summary ---")
print(json.dumps(response["summary"], ensure_ascii=False, indent=2))

In [ ]:
# houses 確認
print("--- houses ---")
print(f"ASC: {response['houses']['asc']:.4f}°")
print(f"MC:  {response['houses']['mc']:.4f}°")
print(f"cusps (H1-H12): {[round(c,2) for c in response['houses']['cusps']]}")

In [ ]:
# アスペクト上位5件
print("--- aspects (orb順 top5) ---")
top5 = sorted(response["aspects"], key=lambda x: x["orb"])[:5]
for a in top5:
    print(f"  {a['planet1_jp']} {a['aspect_jp']} {a['planet2_jp']}  orb={a['orb']}° {a['type']} {a['applying_jp']}")

In [ ]:
# rulers 先頭3件
print("--- rulers (H1-H3) ---")
for r in response["rulers"][:3]:
    print(f"  H{r['house_num']} カスプ:{r['cusp_sign']} → ルーラー:{r['ruler']} ({r['ruler_sign_jp']}, H{r['ruler_house']}) [{r['dignity']}]")    

## 10. JSON シリアライズ検証

FastAPI は dict をそのまま JSON 化する。float / int / str / list / None の型が混在していないか確認。

In [ ]:
import json

try:
    s = json.dumps(response, ensure_ascii=False)
    print(f"JSON シリアライズ成功: {len(s):,} bytes")
except TypeError as e:
    print(f"エラー: {e}")

## 11. 別テストケース：ビヨンセ 1981-09-04 10:00 Houston TX

都市名ジオコーディング機能の確認。

In [ ]:
lat2, lon2, tz2 = engine.geocode_city("Houston, Texas, USA")
print(f"Houston: lat={lat2:.4f}, lon={lon2:.4f}, tz={tz2}")

natal2   = engine.calc_natal(datetime(1981, 9, 4, 10, 0), lat2, lon2, tz2)
summary2 = engine.calc_summary(natal2)

sun2  = natal2["planets"].get("Sun",  {})
moon2 = natal2["planets"].get("Moon", {})
asc2  = natal2["planets"].get("ASC",  {})
print(f"Sun:  {sun2.get('degree_str')}")
print(f"Moon: {moon2.get('degree_str')}")
print(f"ASC:  {asc2.get('degree_str')}")
# 期待値: Sun Virgo, Moon Scorpio

## 12. スキーマ確認チェックリスト

| 項目 | 確認 |
|------|------|
| planets: 全11天体 + ASC/MC/SNode/Fortune | ✅ / ❌ |
| planets: dignity/dignity_score マージ済み | ✅ / ❌ |
| houses: cusps[0..11], asc, mc | ✅ / ❌ |
| aspects: planet1/2, aspect, orb, type | ✅ / ❌ |
| summary: element_count, modality_count, stelliums | ✅ / ❌ |
| rulers: 12ハウス分, dignity付き | ✅ / ❌ |
| total_score / score_label | ✅ / ❌ |
| JSON シリアライズ可 | ✅ / ❌ |

## 次のステップ

1. **scoring.py 作成** — dignity スコアに加え、アスペクトスコア・ハウス位置スコアを組み合わせた 0–100 スコアリング
2. **Pydantic モデル定義** — 上記スキーマを FastAPI `response_model` として型定義
3. **POST /chart エンドポイント実装** — `app.py` に組み込み
4. **POST /synastry** — 2チャート比較アスペクト
5. **GET /transit** — 通過天体計算 (`transits.py` の検証 → 02_transit.ipynb)